# Snowflake Cortex AI SQL: Extracting Insights from Multimodal Customer Service Data

This notebook demonstrates how to build a comprehensive customer service analytics system using **Snowflake Cortex AI functions**. You'll process audio recordings, chat logs, support tickets, and PDF documents - all using pure SQL.

### What You'll Learn
| AI Function | Purpose | Input | Output |
|-------------|---------|-------|--------|
| `AI_TRANSCRIBE` | Speech-to-text | Audio files | Text with speaker segments |
| `AI_TRANSLATE` | Language translation | Any text | English text |
| `AI_SENTIMENT` | Emotion detection | Text | positive/negative/neutral/mixed |
| `AI_CLASSIFY` | Categorization | Text + categories | Best matching category |
| `AI_COMPLETE` | LLM generation | Prompt | Generated text |
| `AI_PARSE_DOCUMENT` | Document extraction | PDF files | Structured content |
| `AI_EXTRACT` | Field extraction | Text + schema | Structured JSON |

### Prerequisites
- Run `setup.sql` before starting this notebook
- Ensure your account has Cortex AI functions enabled
- Use a warehouse with sufficient compute (MEDIUM or larger recommended)

### Snowflake SQL Concepts Used in This Lab

This lab uses several Snowflake SQL patterns you'll see repeatedly. Here's a quick reference:

**VARIANT and `:` notation** — Snowflake stores JSON/semi-structured data in a `VARIANT` column. You access nested fields with `:` instead of dot notation:
```sql
-- Access a field:      my_column:field_name
-- Access nested field: my_column:parent:child
-- Cast to a type:      my_column:field_name::VARCHAR
-- Access array element: my_column:array_field[0]
```

**OBJECT_CONSTRUCT()** — Builds a JSON object from key-value pairs:
```sql
OBJECT_CONSTRUCT('name', 'Alice', 'age', 30)
-- Returns: {"name": "Alice", "age": 30}
```

**TO_FILE()** — Creates a reference to a file stored in a Snowflake stage. Required by `AI_TRANSCRIBE` and `AI_PARSE_DOCUMENT`:
```sql
TO_FILE('@MY_STAGE/path/to/file.mp3')
-- or with two arguments:
TO_FILE('@MY_STAGE', 'relative/path/file.pdf')
```

**FLATTEN()** — Explodes a JSON array into individual rows. Think of it as "unnesting":
```sql
-- If segments = [{"text": "Hello"}, {"text": "World"}]
TABLE(FLATTEN(input => my_column:segments)) seg
-- Produces 2 rows. Access each element via seg.value:text
```

**CTEs (WITH ... AS)** — Common Table Expressions let you break complex queries into named steps:
```sql
WITH step1 AS (
    SELECT ... -- first transformation
),
step2 AS (
    SELECT ... FROM step1 -- builds on step1
)
SELECT ... FROM step2; -- final result
```

**DIRECTORY()** — Lists all files in a stage as a table:
```sql
SELECT * FROM DIRECTORY(@MY_STAGE);
-- Returns: RELATIVE_PATH, SIZE, LAST_MODIFIED, etc.
```

You don't need to memorize these now — each section explains them in context.

## Step 0: Explore the Sample Data

Before processing, let's understand what data we're working with. The setup script created:
- **Audio files**: Customer service call recordings in the `@CUSTOMER_CALLS` stage
- **PDF documents**: Company documents in the `@COMPANY_DOCUMENTS` stage  
- **Chat logs**: Customer chat transcripts with agent classifications
- **Support tickets**: Formal support tickets linked to chats

Run each cell below and check that you see data. If any query returns 0 rows, go back and re-run `setup.sql`.

In [ ]:
-- View available audio files
SELECT * FROM DATA.audio_file_list;

> **Expected:** 5 rows — one per audio file (`.mp3`). These are customer service call recordings that we'll transcribe and analyze.

In [ ]:
-- Preview chat logs structure
SELECT chat_id, customer_name, self_reported_category, self_reported_sentiment, resolution_status
FROM CHAT_LOGS LIMIT 5;

> **Expected:** 5 rows (of 20 total). Notice the `self_reported_category` and `self_reported_sentiment` columns — these were labeled by human agents. Later, we'll use AI to check if they got it right.

In [ ]:
-- Preview support tickets
SELECT ticket_id, ticket_number, subject, self_reported_category, priority, product_affected
FROM SUPPORT_TICKETS LIMIT 5;

> **Expected:** 5 rows (of 20 total). Each ticket is linked to a chat via `ticket_id`. We'll use AI to check whether the ticket description matches what was actually discussed in the chat.

---
## Part 1: Audio Processing Pipeline

### The Complete Pipeline (Production Approach)

This query chains **6 AI functions** together in a single statement to process customer calls:

```
Audio File → AI_TRANSCRIBE → AI_TRANSLATE → AI_SENTIMENT → AI_CLASSIFY → AI_COMPLETE → Results Table
```

**Why chain them?** Each function builds on the previous output:
1. Can't analyze sentiment until we have text (transcription)
2. Can't classify until we have English text (translation)
3. Summaries need the full translated conversation

⏱️ **Expected runtime**: ~30-60 seconds per audio file (depends on length)

> **Don't worry if the query below looks complex!** It uses CTEs (the `WITH ... AS` pattern from the primer above) to chain steps together. We'll break every piece down in the cells that follow.

In [ ]:
INSERT INTO transcription_results (
    stage_location,
    file_name,
    timestamp_granularity,
    audio_duration,
    segments,
    raw_response,
    translated_text,
    sentiment_label,
    call_category,
    call_summary,
    transcription_completed_at
)
WITH transcriptions AS (
    SELECT 
        file_name,
        AI_TRANSCRIBE(
            TO_FILE('@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name),
            OBJECT_CONSTRUCT('timestamp_granularity', 'speaker')
        ) AS trans_result
    FROM data.audio_file_list
),
with_transcripts AS (
    SELECT
        file_name,
        trans_result,
        ARRAY_TO_STRING(
            ARRAY_AGG(seg.value:text::VARCHAR) WITHIN GROUP (ORDER BY seg.index),
            ' '
        ) AS full_transcript
    FROM transcriptions,
         TABLE(FLATTEN(input => trans_result:segments)) seg
    GROUP BY file_name, trans_result
),
with_translation AS (
    SELECT
        file_name,
        trans_result,
        full_transcript,
        AI_TRANSLATE(full_transcript,'' ,'en') AS translated_transcript
    FROM with_transcripts
)
SELECT
    '@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name AS stage_location,
    file_name,
    'speaker' AS timestamp_granularity,
    trans_result:audio_duration::FLOAT AS audio_duration,
    trans_result:segments AS segments,
    trans_result AS raw_response,
    translated_transcript AS translated_text,
    AI_SENTIMENT(translated_transcript):categories[0]:sentiment::VARCHAR AS sentiment_label,
    AI_CLASSIFY(
        translated_transcript,
        [
            OBJECT_CONSTRUCT('label', 'Fraud & Security Issues', 
                             'description', 'Unauthorized transactions, identity theft, account freezes, fraudulent charges'),
            OBJECT_CONSTRUCT('label', 'Technical & System Errors',
                             'description', 'Auto-pay failures, system glitches, login problems, display issues, platform malfunctions'),
            OBJECT_CONSTRUCT('label', 'Payment & Transaction Problems',
                             'description', 'Duplicate charges, failed payments, processing errors, fee disputes, rate increases'),
            OBJECT_CONSTRUCT('label', 'Account Changes & Modifications',
                             'description', 'Investment reallocation, policy modifications, coverage changes, fund transfers, limit adjustments'),
            OBJECT_CONSTRUCT('label', 'General Inquiries & Information Requests',
                             'description', 'Status checks, documentation requests, simple questions, coverage verification, timeline questions')
        ],
        OBJECT_CONSTRUCT('task_description', 'Classify customer service calls into issue categories based on the main problem discussed')
    ):labels[0]::VARCHAR AS call_category,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        CONCAT('Summarize this call in 50 words: ', translated_transcript)
    ) AS call_summary,
    CURRENT_TIMESTAMP() AS transcription_completed_at
FROM with_translation;

> **Expected:** The query inserts 5 rows into `transcription_results` — one per audio file. It takes ~2-3 minutes. While it runs, read ahead to see what each step does.
>
> **What just happened?** In a single SQL statement, Snowflake: transcribed audio to text, detected speakers, translated to English, analyzed sentiment, classified the issue type, and generated a summary. All without leaving SQL.

---
### Breaking Down Each AI Function

The following cells demonstrate each function individually so you can understand what's happening at each step.

## AI_TRANSCRIBE

Converts audio files to text with speaker diarization (identifying who said what).

**Key parameters:**
- `TO_FILE()`: Points to the audio file in a stage
- `timestamp_granularity: 'speaker'`: Groups text by speaker turns

**Output structure:**
```json
{
  "audio_duration": 45.2,
  "segments": [
    {"speaker": "SPEAKER_00", "text": "Hello, how can I help?", "start": 0.0, "end": 2.1},
    {"speaker": "SPEAKER_01", "text": "I have a billing issue...", "start": 2.5, "end": 8.3}
  ]
}
```

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_transcriptions AS
SELECT 
    file_name,
    AI_TRANSCRIBE(
        TO_FILE('@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name),
        OBJECT_CONSTRUCT('timestamp_granularity', 'speaker')
    ) AS trans_result
FROM data.audio_file_list;

SELECT * FROM temp_transcriptions LIMIT 5;

> **Expected:** 5 rows. The `trans_result` column contains a VARIANT (JSON) object. Click on a cell to expand it — you'll see `audio_duration` (in seconds) and a `segments` array with speaker turns.
>
> **Why a temporary table?** We use `CREATE OR REPLACE TEMPORARY TABLE` so the results persist for the next step but are automatically cleaned up when your session ends. This is a common pattern for multi-step SQL pipelines.

### Flattening the Transcript Segments

The transcription returns an array of segments. We use `FLATTEN()` to explode this array and `ARRAY_AGG()` to combine all text into one conversation string.

**Why?** Downstream AI functions (sentiment, classification) work better on complete text rather than fragments.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_transcripts AS
SELECT
    file_name,
    trans_result,
    ARRAY_TO_STRING(
        ARRAY_AGG(seg.value:text::VARCHAR) WITHIN GROUP (ORDER BY seg.index),
        ' '
    ) AS full_transcript
FROM temp_transcriptions,
     TABLE(FLATTEN(input => trans_result:segments)) seg
GROUP BY file_name, trans_result;

SELECT * FROM temp_with_transcripts LIMIT 5;

> **Expected:** 5 rows. Compare `full_transcript` (one long string) to the `trans_result` segments array — `FLATTEN` + `ARRAY_AGG` combined all the speaker segments into a single conversation text.
>
> **How it works:** `TABLE(FLATTEN(...))` creates one row per array element. Then `ARRAY_AGG(...) WITHIN GROUP (ORDER BY seg.index)` reassembles them in order. The `GROUP BY` collapses everything back to one row per file.

## AI_TRANSLATE

Automatically detects the source language and translates to English (or any target language).

**Syntax:** `AI_TRANSLATE(text, source_language, target_language)`
- Use empty string `''` for source to auto-detect
- Use ISO language codes: `'en'`, `'es'`, `'fr'`, `'de'`, etc.

**Use case:** Global customer service teams can analyze all calls in a single language regardless of the original conversation language.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_translation AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    AI_TRANSLATE(full_transcript, '', 'en') AS translated_transcript
FROM temp_with_transcripts;

SELECT * FROM temp_with_translation LIMIT 5;

> **Expected:** 5 rows. Compare `full_transcript` and `translated_transcript` — calls that were already in English stay the same; calls in other languages now have English translations.
>
> **How auto-detect works:** The empty string `''` as the source language tells Snowflake to figure out the language automatically. This is useful when your data contains multiple languages.

## AI_SENTIMENT

Analyzes the emotional tone of text and returns a sentiment classification.

**Output structure:**
```json
{
  "categories": [
    {"sentiment": "negative", "score": 0.85},
    {"sentiment": "neutral", "score": 0.10},
    {"sentiment": "positive", "score": 0.05}
  ]
}
```

**Possible values:** `positive`, `negative`, `neutral`, `mixed`, `very_negative`

**Business use:** Identify frustrated customers for escalation, track sentiment trends over time.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_sentiment AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    AI_SENTIMENT(translated_transcript):categories[0]:sentiment::VARCHAR AS sentiment_label
FROM temp_with_translation;

SELECT * FROM temp_with_sentiment LIMIT 5;

> **Expected:** 5 rows. The `sentiment_label` column shows values like `positive`, `negative`, `neutral`, or `mixed`. Notice we extracted just the top category with `:categories[0]:sentiment` — the full JSON response contains scores for all categories.

## AI_CLASSIFY

Categorizes text into one of your custom-defined categories. This is **zero-shot classification** - no training required!

**Key features:**
- Define your own categories with labels and descriptions
- Descriptions help the AI understand what belongs in each category
- Returns confidence scores for each category

**Our 5 categories:**
1. Fraud & Security Issues
2. Technical & System Errors  
3. Payment & Transaction Problems
4. Account Changes & Modifications
5. General Inquiries & Information Requests

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_classification AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    sentiment_label,
    AI_CLASSIFY(
        translated_transcript,
        [
            OBJECT_CONSTRUCT('label', 'Fraud & Security Issues', 
                             'description', 'Unauthorized transactions, identity theft, account freezes, fraudulent charges'),
            OBJECT_CONSTRUCT('label', 'Technical & System Errors',
                             'description', 'Auto-pay failures, system glitches, login problems, display issues, platform malfunctions'),
            OBJECT_CONSTRUCT('label', 'Payment & Transaction Problems',
                             'description', 'Duplicate charges, failed payments, processing errors, fee disputes, rate increases'),
            OBJECT_CONSTRUCT('label', 'Account Changes & Modifications',
                             'description', 'Investment reallocation, policy modifications, coverage changes, fund transfers, limit adjustments'),
            OBJECT_CONSTRUCT('label', 'General Inquiries & Information Requests',
                             'description', 'Status checks, documentation requests, simple questions, coverage verification, timeline questions')
        ],
        OBJECT_CONSTRUCT('task_description', 'Classify customer service calls into issue categories based on the main problem discussed')
    ):labels[0]::VARCHAR AS call_category
FROM temp_with_sentiment;


SELECT * FROM temp_with_classification LIMIT 5;

> **Expected:** 5 rows. The `call_category` column shows which of our 5 categories the AI assigned. Try expanding the full AI_CLASSIFY response to see confidence scores for each category.
>
> **Why descriptions matter:** Notice each category has both a `label` and a `description`. The descriptions give the AI model context about what belongs in each category — without them, classification accuracy drops significantly.

## AI_COMPLETE

The most flexible function - runs any prompt through an LLM. Here we use it for summarization.

**Syntax:** `AI_COMPLETE(model_name, prompt)`

**Available models:**
- `claude-sonnet-4-5` - High quality, balanced speed/cost (used here)
- `llama3.1-70b` - Open source alternative
- `mistral-large2` - Fast, cost-effective
- See docs for full list

**Pro tip:** Be specific in your prompt about output format and length constraints.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_summary AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    sentiment_label,
    call_category,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        CONCAT('Summarize this call in 50 words: ', translated_transcript)
    ) AS call_summary
FROM temp_with_classification;

SELECT * FROM temp_with_summary LIMIT 5;

> **Expected:** 5 rows. Read the `call_summary` column — each one should be a concise ~50-word summary of the conversation. The prompt `'Summarize this call in 50 words'` controls the output format.
>
> **Prompt engineering tip:** `AI_COMPLETE` is the most flexible function — it can do anything an LLM can do. The quality of the output depends entirely on your prompt. Try changing "50 words" to "3 bullet points" and see what happens.

## Combine All Results

Now we insert all the processed data into our permanent results table. This is the same data the "complex query" at the top would generate, but here we built it step-by-step.

In [ ]:
-- Clear any previous results to avoid duplicates if you also ran the complex pipeline above
TRUNCATE TABLE transcription_results;

INSERT INTO transcription_results (
    stage_location,
    file_name,
    timestamp_granularity,
    audio_duration,
    segments,
    raw_response,
    translated_text,
    sentiment_label,
    call_category,
    call_summary,
    transcription_completed_at
)
SELECT
    '@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name AS stage_location,
    file_name,
    'speaker' AS timestamp_granularity,
    trans_result:audio_duration::FLOAT AS audio_duration,
    trans_result:segments AS segments,
    trans_result AS raw_response,
    translated_transcript AS translated_text,
    sentiment_label,
    call_category,
    call_summary,
    CURRENT_TIMESTAMP() AS transcription_completed_at
FROM temp_with_summary;

## Clean Up Temporary Tables

Remove the intermediate tables we created during the step-by-step demonstration.

In [ ]:
DROP TABLE IF EXISTS temp_transcriptions;
DROP TABLE IF EXISTS temp_with_transcripts;
DROP TABLE IF EXISTS temp_with_translation;
DROP TABLE IF EXISTS temp_with_sentiment;
DROP TABLE IF EXISTS temp_with_classification;
DROP TABLE IF EXISTS temp_with_summary;

## View Final Results

Query the results table to see all processed calls with their transcriptions, translations, sentiments, categories, and AI-generated summaries.

In [ ]:
SELECT * FROM transcription_results;

### Quick Analysis of Results

Let's see the distribution of categories and sentiments across our processed calls.

In [ ]:
SELECT 
    call_category,
    sentiment_label,
    COUNT(*) as call_count,
    ROUND(AVG(audio_duration), 1) as avg_duration_seconds
FROM transcription_results
GROUP BY call_category, sentiment_label
ORDER BY call_count DESC;

---
## Part 2: Document Processing with AI_PARSE_DOCUMENT

Now let's process a different type of unstructured data: **PDF documents**.

`AI_PARSE_DOCUMENT` extracts structured content from PDF files while preserving the document's layout — headers, paragraphs, tables, and formatting are all maintained.

**How it works:**
1. We use `DIRECTORY(@COMPANY_DOCUMENTS)` to list all files in our stage as a table
2. For each file, `TO_FILE()` creates a reference that `AI_PARSE_DOCUMENT` can read
3. The function returns a VARIANT (JSON) containing the extracted content

**Key parameters:**
- `mode: 'LAYOUT'` — Preserves document structure (headers, tables, lists) using markdown formatting. Without this, you'd get plain text with no structure.
- `page_split: TRUE` — Returns content page-by-page as separate array elements. Useful for large documents where you want to process individual pages.

**Use cases:** Policy documents, contracts, invoices, manuals, reports — anything stored as PDF that you need to analyze with AI.

In [ ]:
CREATE TABLE IF NOT EXISTS parsed_documents_raw AS
SELECT 
    RELATIVE_PATH AS file_name,
    '@COMPANY_DOCUMENTS/' || RELATIVE_PATH AS file_path,
    SIZE AS file_size_bytes,
    LAST_MODIFIED,
    AI_PARSE_DOCUMENT(
        TO_FILE('@COMPANY_DOCUMENTS', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'page_split': TRUE}
    ) AS parsed_result
FROM DIRECTORY(@COMPANY_DOCUMENTS);

Select * from parsed_documents_raw limit 10;

> **Expected:** One row per PDF document in the stage. The `parsed_result` column is a VARIANT containing the full extracted content.
>
> The raw output looks like dense JSON. Let's extract something readable from it:

In [ ]:
-- Extract the text content from the first page of each document
SELECT 
    file_name,
    parsed_result:content[0]:content::VARCHAR AS first_page_content
FROM parsed_documents_raw
LIMIT 3;

> **What you're seeing:** The extracted text preserves markdown formatting — `#` for headers, `|` for table rows, `-` for lists. This structural information is valuable: it means downstream AI functions can understand *what part* of the document they're reading (a heading vs. a table cell vs. body text).
>
> **Why this matters:** Traditional PDF-to-text tools lose all structure and produce a flat string. `AI_PARSE_DOCUMENT` with `LAYOUT` mode preserves the document's semantic structure, which dramatically improves the quality of any downstream analysis.

---
## Part 3: Validating Chat Logs with AI

Customer service agents manually classify chats during or after the conversation — assigning a category (like "Billing" or "Technical Support") and noting the customer's sentiment. But humans make mistakes, especially under time pressure.

**The business problem:** How do you check the accuracy of thousands of manual classifications? You can't re-read every chat. But AI can.

The query below runs **3 AI functions** on each of the 20 chat conversations:
1. `AI_CLASSIFY` — Re-classifies the conversation into categories
2. `AI_SENTIMENT` — Re-analyzes the emotional tone
3. `AI_EXTRACT` — Pulls structured fields (issue description, product name, error codes, etc.) from unstructured chat text

Then it **compares** the AI results against the human agent's labels and **flags mismatches**.

### New function: AI_EXTRACT

This function pulls specific fields from unstructured text based on a schema you define:

```sql
AI_EXTRACT(
    text => 'your text here',
    responseFormat => OBJECT_CONSTRUCT(
        'field_name', 'Description of what to extract',
        'another_field', 'Another question to answer from the text'
    )
)
```

Think of it as giving the AI a form to fill out by reading the text. Each key is a field name, each value is a question/instruction.

### How the query is structured

The query uses 3 CTEs (named steps):

| CTE | What it does |
|-----|-------------|
| `conversation_prep` | Flattens the JSON chat messages into a single text string per conversation (same `FLATTEN` + `ARRAY_AGG` pattern we used for audio segments) |
| `ai_analysis` | Runs `AI_CLASSIFY`, `AI_SENTIMENT`, and `AI_EXTRACT` on each conversation |
| Final `SELECT` | Compares agent labels vs. AI labels, normalizes sentiment values, and sets `is_flagged = TRUE` when there's a mismatch |

**Flagging logic (in plain English):**
- Flag if the agent's category differs from what AI classified
- Flag if the agent's sentiment label doesn't match AI's sentiment
- Always flag `mixed` sentiment (needs human review)

⏱️ **Expected runtime:** ~1-2 minutes (20 chats × 3 AI functions each)

In [ ]:
CREATE OR REPLACE TABLE chat_validation_results AS
WITH conversation_prep AS (
    SELECT 
        c.chat_id,
        c.ticket_id,
        c.customer_name,
        c.agent_name,
        c.chat_timestamp,
        c.self_reported_category,
        c.self_reported_sentiment,
        c.resolution_status,
        ARRAY_TO_STRING(
            ARRAY_AGG(msg.value:message::VARCHAR) 
            WITHIN GROUP (ORDER BY msg.index), 
            ' '
        ) AS full_conversation
    FROM chat_logs c,
    LATERAL FLATTEN(input => PARSE_JSON(c.messages)) msg
    GROUP BY 
        c.chat_id, c.ticket_id, c.customer_name, c.agent_name,
        c.chat_timestamp, c.self_reported_category, 
        c.self_reported_sentiment, c.resolution_status
),
ai_analysis AS (
    SELECT 
        chat_id,
        ticket_id,
        customer_name,
        agent_name,
        chat_timestamp,
        self_reported_category,
        self_reported_sentiment,
        resolution_status,
        full_conversation,
        AI_CLASSIFY(
            full_conversation,
            ARRAY_CONSTRUCT('Technical Support', 'Bug Report', 'Feature Request', 'Billing', 'General Inquiry')
        ) AS ai_classify_object,
        AI_SENTIMENT(full_conversation) AS ai_sentiment_object,
        AI_EXTRACT(
            text => full_conversation,
            responseFormat => OBJECT_CONSTRUCT(
                'issue_description', 'What is the main issue or problem described?',
                'product_name', 'What product is mentioned?',
                'error_message', 'What error message or code is mentioned?',
                'resolution_provided', 'What solution or resolution was provided?',
                'customer_satisfaction_indicator', 'Does the customer seem satisfied?',
                'urgency_level', 'How urgent is this issue?'
            )
        ) AS extracted_features
    FROM conversation_prep
)
SELECT 
    chat_id,
    ticket_id,
    customer_name,
    agent_name,
    chat_timestamp,
    self_reported_category,
    self_reported_sentiment,
    resolution_status,
    full_conversation,
    ai_classify_object:labels[0]::VARCHAR AS ai_classified_category,
    ai_sentiment_object:categories[0]:sentiment::VARCHAR AS ai_sentiment_raw,
    CASE 
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' THEN 'Mixed'
        ELSE 'Unknown'
    END AS ai_sentiment_normalized,
    extracted_features,
    CASE 
        WHEN self_reported_category != ai_classify_object:labels[0]::VARCHAR 
         AND ai_classify_object:labels[0]::VARCHAR IS NOT NULL THEN TRUE
        WHEN self_reported_sentiment != CASE 
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
                ELSE 'Unknown'
            END
         AND NOT (self_reported_sentiment = 'Frustrated' 
                  AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative'))
         AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IS NOT NULL THEN TRUE
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' THEN TRUE
        ELSE FALSE
    END AS is_flagged,
    ARRAY_CONSTRUCT_COMPACT(
        CASE 
            WHEN self_reported_category != ai_classify_object:labels[0]::VARCHAR
             AND ai_classify_object:labels[0]::VARCHAR IS NOT NULL
            THEN 'category_mismatch'
        END,
        CASE 
            WHEN self_reported_sentiment != CASE 
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
                    ELSE 'Unknown'
                END
             AND NOT (self_reported_sentiment = 'Frustrated' 
                      AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative'))
             AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IS NOT NULL
            THEN 'sentiment_mismatch'
        END,
        CASE 
            WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' 
           THEN 'mixed_sentiment'
        END
    ) AS flag_reasons,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ai_analysis;

Select * from chat_validation_results LIMIT 5;

> **Expected:** 20 rows (one per chat). Key columns to look at:
> - `self_reported_category` vs `ai_classified_category` — Do they match?
> - `self_reported_sentiment` vs `ai_sentiment_normalized` — Does the AI agree with the agent?
> - `is_flagged` — `TRUE` means AI found a discrepancy
> - `flag_reasons` — An array explaining *why* it was flagged (e.g., `category_mismatch`, `sentiment_mismatch`)
> - `extracted_features` — Click to expand the JSON and see the structured data AI pulled from the conversation

### Analyzing Validation Results

Let's see how many chats were flagged for discrepancies and what types of issues were found.

> This tells you what percentage of your agents' classifications the AI disagrees with — a useful QA metric.

In [ ]:
SELECT 
    is_flagged,
    COUNT(*) as chat_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as percentage
FROM chat_validation_results
GROUP BY is_flagged;

> **What to look for:** The percentage of flagged chats tells you how accurate your human agents are. In a real system, you'd use this to identify agents who need additional training, or categories that are frequently confused.

---
## Part 4: Cross-Validating Tickets and Chats

The final analysis uses `AI_COMPLETE` for a complex task: **semantic comparison** between support tickets and their linked chat conversations.

**Why this matters:**
- Tickets are often created *after* chats, sometimes by different people
- Information can be lost or misrepresented in translation
- Identifying misalignments helps improve data quality and customer outcomes

**What we check:**
- Is the core issue the same?
- Is the product/service consistent?
- Are categories aligned?

### How the query works

| CTE | What it does |
|-----|-------------|
| `ticket_chat_data` | JOINs tickets to their linked chats (via `ticket_id`) to get both sides of the story in one row |
| `ai_responses` | Sends both the ticket and chat to `AI_COMPLETE` with a detailed prompt asking it to compare them |
| Final `SELECT` | Parses the AI's JSON response and creates flags for misalignments |

### Prompt engineering patterns used

This query demonstrates several important `AI_COMPLETE` techniques:

- **Structured prompt with `||`** — We build the prompt by concatenating SQL columns with template text. This lets us feed dynamic data (ticket subject, chat conversation) into a fixed prompt template.
- **JSON output format** — We tell the model exactly what JSON structure to return: `{"alignment": "...", "confidence": "...", ...}`. This makes the output parseable with `PARSE_JSON()`.
- **Low temperature (0.1)** — The `model_parameters => OBJECT_CONSTRUCT('temperature', 0.1)` makes the model more deterministic and consistent. For analytical tasks (vs. creative writing), low temperature is preferred.
- **REGEXP_REPLACE for cleanup** — LLMs sometimes wrap JSON in markdown code fences (` ```json ... ``` `). The regex strips these before parsing.

⏱️ **Expected runtime:** ~1-2 minutes (20 ticket-chat pairs)

In [ ]:
CREATE OR REPLACE TABLE ticket_chat_alignment AS
WITH ticket_chat_data AS (
    SELECT 
        t.ticket_id,
        t.ticket_number,
        t.customer_name AS ticket_customer_name,
        t.subject AS ticket_subject,
        t.description AS ticket_description,
        t.self_reported_category AS ticket_category,
        t.priority AS ticket_priority,
        t.status AS ticket_status,
        t.product_affected AS ticket_product,
        cv.chat_id,
        cv.customer_name AS chat_customer_name,
        cv.self_reported_category AS chat_category,
        cv.self_reported_sentiment AS chat_sentiment,
        c.product_mentioned AS chat_product,
        cv.full_conversation
    FROM support_tickets t
    INNER JOIN chat_validation_results cv ON t.ticket_id = cv.ticket_id
    INNER JOIN chat_logs c ON cv.chat_id = c.chat_id
),
ai_responses AS (
    SELECT 
        ticket_id,
        ticket_number,
        chat_id,
        ticket_customer_name,
        chat_customer_name,
        ticket_category,
        chat_category,
        ticket_product,
        chat_product,
        ticket_subject,
        ticket_description,
        full_conversation,
        AI_COMPLETE(
            model => 'claude-sonnet-4-5',
            prompt => 'Compare the ticket and chat to determine if they discuss the same issue.\n\n' ||
                'TICKET:\n' ||
                'Subject: ' || ticket_subject || '\n' ||
                'Description: ' || ticket_description || '\n' ||
                'Category: ' || ticket_category || '\n' ||
                'Product: ' || COALESCE(ticket_product, 'Not specified') || '\n\n' ||
                'CHAT:\n' ||
                'Conversation: ' || full_conversation || '\n' ||
                'Category: ' || chat_category || '\n' ||
                'Product: ' || COALESCE(chat_product, 'Not mentioned') || '\n\n' ||
                'Analyze if the ticket and chat are about the SAME core issue. Consider:\n' ||
                '1. Is the main problem/request the same?\n' ||
                '2. Is the product/service the same?\n' ||
                '3. Is the technical issue consistent?\n\n' ||
                'Respond ONLY with valid JSON (no markdown formatting):\n' ||
                '{"alignment":"aligned|misaligned|partial","confidence":"high|medium|low",' ||
                '"reason":"brief explanation","severity":"critical|moderate|minor"}',
            model_parameters => OBJECT_CONSTRUCT('temperature', 0.1)
        ) AS ai_alignment_response_raw,
        REGEXP_REPLACE(
            REGEXP_REPLACE(ai_alignment_response_raw, '^```json\\s*', ''),
            '\\s*```$', 
            ''
        ) AS ai_alignment_response
    FROM ticket_chat_data
)
SELECT 
    ticket_id,
    ticket_number,
    chat_id,
    ticket_customer_name,
    chat_customer_name,
    ticket_category,
    chat_category,
    ticket_product,
    chat_product,
    ticket_subject,
    ticket_description,
    full_conversation,
    ai_alignment_response,
    PARSE_JSON(ai_alignment_response):alignment::VARCHAR AS alignment_status,
    PARSE_JSON(ai_alignment_response):confidence::VARCHAR AS alignment_confidence,
    PARSE_JSON(ai_alignment_response):reason::VARCHAR AS alignment_reason,
    PARSE_JSON(ai_alignment_response):severity::VARCHAR AS misalignment_severity,
    CASE 
        WHEN ticket_category = chat_category THEN FALSE
        ELSE TRUE
    END AS category_mismatch_flag,
    CASE 
        WHEN ticket_product IS NOT NULL 
         AND chat_product IS NOT NULL
         AND LOWER(ticket_product) != LOWER(chat_product) 
        THEN TRUE
        ELSE FALSE
    END AS product_mismatch_flag,
    CASE 
        WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'misaligned' THEN TRUE
        WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'partial' 
         AND PARSE_JSON(ai_alignment_response):severity::VARCHAR IN ('critical', 'moderate') THEN TRUE
        WHEN ticket_category != chat_category THEN TRUE
        WHEN ticket_product IS NOT NULL 
         AND chat_product IS NOT NULL
         AND LOWER(ticket_product) != LOWER(chat_product) THEN TRUE
        ELSE FALSE
    END AS is_flagged,
    ARRAY_CONSTRUCT_COMPACT(
        CASE 
            WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'misaligned' 
            THEN 'issue_misalignment'
        END,
        CASE 
            WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'partial' 
            THEN 'partial_alignment'
        END,
        CASE 
            WHEN ticket_category != chat_category 
            THEN 'category_mismatch'
        END,
        CASE 
            WHEN ticket_product IS NOT NULL 
             AND chat_product IS NOT NULL
             AND LOWER(ticket_product) != LOWER(chat_product)
            THEN 'product_mismatch'
        END
    ) AS flag_reasons,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ai_responses;

SELECT * FROM ticket_chat_alignment LIMIT 5;

> **Expected:** 20 rows. Key columns:
> - `alignment_status` — `aligned`, `misaligned`, or `partial`
> - `alignment_confidence` — `high`, `medium`, or `low`
> - `alignment_reason` — The AI's explanation of why it thinks they match or don't
> - `misalignment_severity` — `critical`, `moderate`, or `minor`
> - `is_flagged` — `TRUE` if there's a meaningful discrepancy worth investigating
>
> **Try this:** Read the `alignment_reason` column — the AI provides a natural language explanation of its comparison. This is the power of `AI_COMPLETE`: it doesn't just classify, it *reasons*.

### Alignment Results Summary

Let's see how well tickets and chats align, and what types of discrepancies were found.

> This breakdown shows how consistent your ticket creation process is. High misalignment rates suggest the ticket creation workflow needs improvement.

In [ ]:
SELECT 
    alignment_status,
    alignment_confidence,
    COUNT(*) as count
FROM ticket_chat_alignment
GROUP BY alignment_status, alignment_confidence
ORDER BY count DESC;

> **What to look for:** Ideally, most rows should be `aligned` with `high` confidence. If you see many `misaligned` or `partial` results, that signals a systemic issue in how tickets are created from chat conversations.

---
## Conclusion

You've built a complete multimodal customer service analytics system using Snowflake Cortex AI functions!

### Key Takeaways

| Function | Best For |
|----------|----------|
| `AI_TRANSCRIBE` | Converting audio/video to searchable text |
| `AI_TRANSLATE` | Multilingual data normalization |
| `AI_SENTIMENT` | Customer satisfaction monitoring |
| `AI_CLASSIFY` | Zero-shot categorization with custom labels |
| `AI_COMPLETE` | Flexible LLM tasks (summaries, comparisons, generation) |
| `AI_PARSE_DOCUMENT` | Structured extraction from PDFs |
| `AI_EXTRACT` | Pulling specific fields from unstructured text |

### Next Steps
- **Scale up**: Process your own audio files and documents
- **Customize categories**: Modify AI_CLASSIFY categories for your business
- **Build dashboards**: Use the results tables for Streamlit or BI tools
- **Automate**: Schedule these queries with Tasks for continuous processing

### Resources
- [Cortex AI Functions Documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [Snowflake Notebooks Guide](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks)